# RG3 F1: 현재 샷 공정 비중/단위 배율 불변 비율

과거센서없이 계산되는시간분율·압력관계·RPM/배압 logratio 소수배율불변표현 및 상호작용8개. InjectionTime-FillingTime은차이proxy이며 정확한보압시간이라고 주장하지 않음. log10비율−round(log10비율)은10의정수배 단위차에 수학적으로불변이지만 실제RPM/압력단위를고친것이아니며 필드정의검증 필요. 실제가스원인/에너지계산이라고 주장하지 않음.

RG3 RH620(정상601/가스19), 같은7개구간. 새전문가RF/ET2시드평균 또는CB/LGBM/XGB3모델평균. 106번CORAL3부스팅 전문가(학습CN7보조포함)와조합및threshold 내부F1 선택. 새로운phase 전문가학습은RG3 ref만. 평가구간은결측/분류/조합/임계값학습제외. 새독립검증아닌개발용중첩검증. 범위/분모변경없음.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
z=np.load(t/'rg3_CORAL_boost_fold_cache.npz',allow_pickle=False)
assert len(d)==620 and np.array_equal(y,z['labels']) and np.array_equal(d.TimeStamp.astype(str),z['timestamps'])
groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
def ratio(a,b):return a/b.where(b.abs()>1e-6)
def mantissa(a,b):
 v=np.log10(ratio(a.abs().where(a.abs()>1e-6),b.abs()))
 return v-np.round(v)
F=pd.DataFrame()
F['fill_fraction']=ratio(d.Filling_Time,d.Cycle_Time)
F['injection_fill_difference_fraction']=ratio(d.Injection_Time-d.Filling_Time,d.Cycle_Time)
F['recovery_fraction']=ratio(d.Plasticizing_Time,d.Cycle_Time)
F['pressure_support_fraction']=ratio(d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure,d.Max_Injection_Pressure.abs()+d.Max_Switch_Over_Pressure.abs())
F['rpm_ratio_log_mantissa']=mantissa(d.Average_Screw_RPM,d.Max_Screw_RPM)
F['back_pressure_ratio_log_mantissa']=mantissa(d.Average_Back_Pressure,d.Max_Back_Pressure)
F['fill_pressure_interaction']=F.fill_fraction*F.pressure_support_fraction
F['recovery_pressure_interaction']=F.recovery_fraction*F.back_pressure_ratio_log_mantissa
F=F.replace([np.inf,-np.inf],np.nan)
F.assign(TimeStamp=times).to_csv(t/'rg3_instant_phase_proxy_features.csv',index=False,encoding='utf-8-sig')
memo={};audit=[]
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  normal=ref[y[ref]==0]
  im=SimpleImputer(strategy='median',keep_empty_features=True).fit(F.iloc[normal]);all_x=im.transform(F)
  forest=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    forest.append(m.fit(all_x[ref],y[ref]).predict_proba(all_x)[:,1])
  boost=[]
  models=[
   CatBoostClassifier(iterations=150,depth=2,learning_rate=.04,l2_leaf_reg=10,auto_class_weights='Balanced',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False),
   LGBMClassifier(n_estimators=150,num_leaves=5,max_depth=2,min_child_samples=15,learning_rate=.03,reg_lambda=10,class_weight='balanced',random_state=42,n_jobs=2,verbosity=-1),
   XGBClassifier(n_estimators=150,max_depth=2,min_child_weight=5,learning_rate=.03,reg_lambda=10,subsample=.8,colsample_bytree=.8,scale_pos_weight=float((y[ref]==0).sum()/y[ref].sum()),random_state=42,n_jobs=2,eval_metric='logloss')]
  for m in models:boost.append(m.fit(all_x[ref],y[ref]).predict_proba(all_x)[:,1])
  reference=z['scores_'+sources[key]][:,2]
  memo[key]=np.column_stack([reference,np.mean(forest,axis=0),np.mean(boost,axis=0)])
  audit.append(dict(train=len(ref),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
mixes=[np.array([a,b,4-a-b],float)/4 for a in range(5) for b in range(5-a)]
records=[];choices=[]
for block in blocks:
 q=np.flatnonzero(groups==block);ref=np.flatnonzero(groups!=block);out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in mixes:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_phase_proxy',best[5],best[4])]
 for col,name in enumerate(['CORAL3_boost_reference','instant_phase_forest','instant_phase_boost']):
  cut,_=cutfit(y[ref],inner[:,col]);named.append((name+'_inner_F1_cut',out[:,col],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(block),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(block),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
 (t/'rg3_instant_phase_progress.json').write_text(str(int(block)),encoding='utf-8')
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_instant_phase_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


                              model       AP  TP  FP  FN  TN       F1   recall
CORAL3_boost_reference_inner_F1_cut 0.140730   7  36  12 565 0.225806 0.368421
      inner_F1_selected_phase_proxy 0.140713   7  36  12 565 0.225806 0.368421
   instant_phase_boost_inner_F1_cut 0.020673  16 530   3  71 0.056637 0.842105
  instant_phase_forest_inner_F1_cut 0.021670  15 543   4  58 0.051993 0.789474
